# Audit & Auth Log Sanitization

This notebook sanitizes **Linux `audit.log`** and **`auth.log`-style syslog** files while preserving security-relevant semantics.

## Policy implemented
- **Users:** Only pseudonymize the username **`newt`** (case-insensitive). All other names are left unchanged.
- **FQDNs:** Only pseudonymize **Azure / Azure-hosted** domains (suffix allowlist). All other FQDNs remain real.
- **IPs / ports / event types / timestamps:** preserved.
- **Stability:** tokens are deterministic under a locally stored secret salt, and mappings are persisted in `mappings/`.

> Tip: If you already generated noisy mappings from previous runs, use the *Reset mappings* cell below once.


In [2]:
from pathlib import Path
import os
import re
import json
import hashlib
import secrets
from typing import Dict, Tuple, Optional, Callable
from collections import defaultdict

# ------------------------
# Configuration
# ------------------------
BASE_DIR = Path.cwd().parent.resolve().joinpath("data")

# Input files
FILES = {
    "audit": "audit.log",
    "auth": "auth.log"
}

def extract_scenario(path: Path) -> str:
    """
    Extract scenario id like 'sc1' from any part of the path.
    Falls back to 'unknown' if not found.
    """
    for part in path.parts:
        if part.startswith("sc") and part[2:].isdigit():
            return part
    return "unknown"

def collect_inputs(base: Path, files: dict) -> dict:
    """
    Return:
      inputs[scenario][table] -> list[Path]  (can be 1 or many)
    We keep lists to support multiple occurrences per scenario if they exist.
    """
    inputs = defaultdict(lambda: defaultdict(list))
    for table, fname in files.items():
        for p in base.rglob(fname):
            sc = extract_scenario(p)
            inputs[sc][table].append(p)
    return inputs

INPUTS = collect_inputs(BASE_DIR, FILES)

# Quick sanity print: how many matches per scenario/table
for sc in sorted(INPUTS):
    counts = {t: len(ps) for t, ps in INPUTS[sc].items()}
    print(f"{sc}: {counts}")

OUT_DIR = BASE_DIR.parent.resolve().joinpath("sanidata")
MAP_DIR = BASE_DIR.parent.resolve().joinpath("mappings") 
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

SALT_PATH = MAP_DIR / "salt.txt"  # stored locally; do NOT publish
ONLY_PSEUDONYMIZE_USERS = {"newt"}     # lowercase allowlist

# Syslog hostname (2nd token) is often environment-specific.
# Keep True if you want to hide the machine name (recommended for privacy).
PSEUDONYMIZE_SYSLOG_HOST = True

# Azure/Azure-hosted FQDN suffix allowlist (suffix match == wildcard *.suffix)
AZURE_FQDN_SUFFIXES = (
    "trafficmanager.net",
    "azure.com",
    "azure.net",
    "azureedge.net",
    "cloudapp.azure.com",
    "azurewebsites.net",
    "windows.net",
    "database.windows.net",
    "redis.cache.windows.net",
    "servicebus.windows.net",
    "vault.azure.net",
    "azure-api.net",
    "core.windows.net",  # optional; covered by windows.net but kept for clarity
)

def _is_empty(x) -> bool:
    return x is None or (isinstance(x, str) and x.strip() == "")


sc3: {'audit': 1, 'auth': 1}
sc4: {'audit': 1, 'auth': 1}
sc7: {'audit': 1}


## (Optional) Reset mappings
Run **once** if your existing `mappings/user_map.json` has polluted keys (e.g., `security`, `taskhostw.exe`, etc.).


In [3]:
# Uncomment and run once to reset mappings.
# for fname in ["user_map.json", "host_map.json", "fqdn_map.json"]:
#     p = MAPPINGS_DIR / fname
#     if p.exists():
#         p.unlink()
# print("Mappings reset.")


## Stable mapping primitives
We keep a secret salt locally and derive compact deterministic tokens. Mappings are persisted per category.


In [4]:
def _load_or_create_salt() -> bytes:
    if SALT_PATH.exists():
        return SALT_PATH.read_bytes()
    salt = secrets.token_bytes(32)
    SALT_PATH.write_bytes(salt)
    return salt

SALT = _load_or_create_salt()

def _mapping_path(map_name: str) -> Path:
    return MAP_DIR / f"{map_name}.json"

def load_mapping(map_name: str) -> Dict[str, str]:
    p = _mapping_path(map_name)
    if not p.exists():
        return {}
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        return {}

def save_mapping(map_name: str, mapping: Dict[str, str]) -> None:
    p = _mapping_path(map_name)
    p.write_text(json.dumps(mapping, indent=2, sort_keys=True), encoding="utf-8")

def map_token(value: str, prefix: str, map_name: str, digits: int = 3) -> str:
    """Deterministic tokenization with collision resolution.

    Token format: PREFIX_XXX (zero-padded numeric id).
    """
    if _is_empty(value):
        return value

    v_norm = str(value)
    mapping = load_mapping(map_name)
    if v_norm in mapping:
        return mapping[v_norm]

    # collision-resistant deterministic id: hash(salt||value||counter) % 10^digits
    mod = 10 ** digits
    counter = 0
    used = set(mapping.values())
    while True:
        h = hashlib.sha256(SALT + v_norm.encode("utf-8") + counter.to_bytes(2, "big")).digest()
        num = int.from_bytes(h[:8], "big") % mod
        token = f"{prefix}_{num:0{digits}d}"
        if token not in used:
            mapping[v_norm] = token
            save_mapping(map_name, mapping)
            return token
        counter += 1


## Sanitizers
The key design choice is **allowlist-only user mapping** (`newt`), and **Azure-only** FQDN mapping.


In [6]:
# ------------------------
# User sanitization (allowlist-only)
# ------------------------
def sanitize_user_token(user: str) -> str:
    """ Pseudonymize only allowlisted usernames (case-insensitive).
    All other values are returned unchanged.
    Supports DOMAIN\\USER by mapping only USER if allowlisted

    """
    if _is_empty(user):
        return user
    u = str(user).strip()

    if "\\" in u:
        dom, name = u.rsplit("\\", 1)
        name_norm = name.strip().lower()
        if name_norm in ONLY_PSEUDONYMIZE_USERS:
            return dom + "\\" + map_token(name_norm, "USER", "user_map", digits=3)
        return u

    name_norm = u.lower()
    if name_norm in ONLY_PSEUDONYMIZE_USERS:
        return map_token(name_norm, "USER", "user_map", digits=3)
    return u

# Whole-word match for 'newt' (safe because allowlist contains only 'newt')
NEWT_WORD_RE = re.compile(r"(?i)\bnewt\b")

def sanitize_user_blob(text: str) -> str:
    """Sanitize allowlisted usernames in free-text contexts."""
    if _is_empty(text):
        return text
    s = str(text)

    # Contextual replacements (e.g., auth.log patterns)
    # Replace only the captured username token; sanitize_user_token enforces allowlist.
    ctx_patterns = [
        re.compile(r"(?i)\b(Invalid user\s+)([^\s]+)"),
        re.compile(r"(?i)\b(Failed password for\s+)([^\s]+)"),
        re.compile(r"(?i)\b(for user\s+)([^\s]+)"),
        re.compile(r"(?i)\b(of user\s+)([^\s]+)"),
        re.compile(r"(?i)\b(User:\s+)([^\s]+)"),
        re.compile(r"(?i)\b(by\s+)([^\s]+)(\(uid=\d+\))"),  # by gdm(uid=0)
    ]

    def _ctx_repl(m):
        # prefix + username [+ optional suffix]
        if len(m.groups()) == 2:
            return m.group(1) + sanitize_user_token(m.group(2))
        return m.group(1) + sanitize_user_token(m.group(2)) + m.group(3)

    for p in ctx_patterns:
        s = p.sub(_ctx_repl, s)

    # Whole-word fallback: replace standalone 'newt'
    s = NEWT_WORD_RE.sub(lambda _: sanitize_user_token("newt"), s)
    return s

# ------------------------
# Hostname / FQDN sanitization
# ------------------------
IP_RE = re.compile(r"^(?:\d{1,3}\.){3}\d{1,3}$")

def sanitize_host_token(host: str) -> str:
    if _is_empty(host):
        return host
    h = str(host).strip()
    # Avoid mapping obvious placeholders or IPs
    if h in {"?", "-"} or IP_RE.match(h):
        return h
    return map_token(h, "HOST", "host_map", digits=3)

def is_azure_fqdn(fqdn: str) -> bool:
    if _is_empty(fqdn):
        return False
    f = str(fqdn).strip().lower().rstrip(".")
    return any(f == sfx or f.endswith("." + sfx) for sfx in AZURE_FQDN_SUFFIXES)

# Very lightweight FQDN-ish matcher (good enough for logs; avoids URLs)
FQDN_RE = re.compile(r"(?i)\b([a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?(?:\.[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?)+)\b")

def sanitize_fqdn_in_text(text: str) -> str:
    if _is_empty(text):
        return text
    s = str(text)

    def _repl(m):
        fqdn = m.group(1)
        f_norm = fqdn.lower().rstrip(".")
        if is_azure_fqdn(f_norm):
            tok = map_token(f_norm, "FQDN", "fqdn_map", digits=4)
            return f"{tok}.example"
        return fqdn

    return FQDN_RE.sub(_repl, s)

# ------------------------
# Path-embedded usernames (only matters for allowlist users; safe)
# ------------------------
WIN_USER_RE = re.compile(r"(?i)(\\users\\)([^\\/]+)(\\)")
NIX_USER_RE = re.compile(r"(?i)(/home/)([^\\/]+)(/)")

def sanitize_user_in_paths(text: str) -> str:
    if _is_empty(text):
        return text
    s = str(text)

    s = WIN_USER_RE.sub(lambda m: m.group(1) + sanitize_user_token(m.group(2)) + m.group(3), s)
    s = NIX_USER_RE.sub(lambda m: m.group(1) + sanitize_user_token(m.group(2)) + m.group(3), s)
    return s


## Line-level sanitization
We sanitize **line-by-line** to preserve ordering and avoid relying on a fixed schema.


In [9]:
# ------------------------
# audit.log line sanitization
# ------------------------
# In audit lines, usernames often appear as acct=... or UID="..." or inside msg='...'
# acct=admin | acct="admin" | acct='admin'
AUDIT_ACCT_RE = re.compile(r"""(?i)\bacct=(?:"([^"]*)"|'([^']*)'|([^\s'"]+))""")

# UID="root" | UID=root | AUID="unset" | AUID=4294967295
AUDIT_UID_RE  = re.compile(r"""(?i)\b(UID|AUID)=(?:"([^"]*)"|'([^']*)'|([^\s'"]+))""")

# hostname=? | hostname=Dev-Linux | hostname=159.223.15.93
AUDIT_HOST_RE = re.compile(r"""(?i)\bhostname=([^\s'"]+)""")

def _strip_quotes(x):
    """
    Return (q1, core, q2) where q1/q2 are quote chars if present.
    Robust to None.
    """
    if x is None:
        return "", "", ""
    x = str(x)

    if len(x) >= 2 and x[0] == '"' and x[-1] == '"':
        return '"', x[1:-1], '"'
    if len(x) >= 2 and x[0] == "'" and x[-1] == "'":
        return "'", x[1:-1], "'"
    return "", x, ""

def _first_non_empty(*vals):
    """Return the first value that is not None and not empty string."""
    for v in vals:
        if v is not None and str(v) != "":
            return v
    return ""

def sanitize_audit_line(line: str) -> str:
    if _is_empty(line):
        return line
    s = str(line)

    # 1) sanitize usernames embedded in file paths (only affects allowlist users)
    s = sanitize_user_in_paths(s)

    # 2) sanitize acct=... occurrences (allowlist-only)
    def _acct_repl(m):
        raw = m.group(1)
        q1, core, q2 = _strip_quotes(raw)
        # Only map if core is allowlisted
        return "acct=" + q1 + sanitize_user_token(core) + q2

    s = AUDIT_ACCT_RE.sub(_acct_repl, s)

    # 3) sanitize UID/AUID string forms (e.g., UID="root") -> only maps if value is 'newt'
    def _uid_repl(m):
        k = m.group(1)  # UID or AUID
        raw = _first_non_empty(m.group(2), m.group(3), m.group(4))
        q1, core, q2 = _strip_quotes(raw)
        # newt-only policy: this will only change if core == "newt" (case-insensitive)
        return f"{k}=" + q1 + sanitize_user_token(core) + q2

    s = AUDIT_UID_RE.sub(_uid_repl, s)

    # 4) hostname=... in msg='...' may be '?' or IP; map only real hostnames
    def _host_repl(m):
        host = m.group(1)
        return "hostname=" + sanitize_host_token(host)

    s = AUDIT_HOST_RE.sub(_host_repl, s)

    # 5) Azure FQDNs in free text (rare in audit, but kept for consistency)
    s = sanitize_fqdn_in_text(s)

    # 6) Free-text safe user contexts (just in case)
    s = sanitize_user_blob(s)

    return s

# ------------------------
# auth.log (syslog-like) line sanitization
# ------------------------
# Example:
# 2025-12-07T07:56:41.233292-05:00 Dev-Linux sshd[7573]: Connection closed by 194.0.234.20 port 65105

def sanitize_auth_line(line: str) -> str:
    if _is_empty(line):
        return line
    s = str(line).rstrip("\n")

    # Split into: timestamp, host, remainder (best-effort)
    parts = s.split(" ", 2)
    if len(parts) < 3:
        # fallback: still apply text-level sanitizers
        s2 = sanitize_user_in_paths(s)
        s2 = sanitize_user_blob(s2)
        s2 = sanitize_fqdn_in_text(s2)
        return s2

    ts, host, rest = parts[0], parts[1], parts[2]

    if PSEUDONYMIZE_SYSLOG_HOST:
        host = sanitize_host_token(host)

    rest = sanitize_user_in_paths(rest)
    rest = sanitize_user_blob(rest)
    rest = sanitize_fqdn_in_text(rest)

    return f"{ts} {host} {rest}"


## Run the sanitization
This will read `audit.log` and `auth.log` (paths configured at the top) and write sanitized outputs.


In [10]:
from pathlib import Path

def output_path_for(input_path: Path, scenario: str, out_root: Path) -> Path:
    """
    Build an output path mirroring the input filename under:
      out_root/<scenario>/<original_filename>.sanitized
    Example:
      data/sc1/audit.log -> sanidata/sc1/audit.log
    """
    sc_dir = out_root / scenario
    sc_dir.mkdir(parents=True, exist_ok=True)
    return sc_dir / input_path.name

def sanitize_file(in_path: Path, out_path: Path, line_sanitizer) -> int:
    """Sanitize a text file line-by-line. Returns number of lines processed."""
    n = 0
    with in_path.open("r", encoding="utf-8", errors="replace") as fin, \
         out_path.open("w", encoding="utf-8") as fout:
        for line in fin:
            fout.write(line_sanitizer(line.rstrip("\n")) + "\n")
            n += 1
    return n

total_audit = 0
total_auth = 0

# Iterate scenarios and sanitize all matched files per scenario.
for sc in sorted(INPUTS.keys()):
    # AUDIT
    for in_audit in INPUTS[sc].get("audit", []):
        out_audit = output_path_for(in_audit, sc, OUT_DIR)
        n = sanitize_file(in_audit, out_audit, sanitize_audit_line)
        total_audit += n
        print(f"[{sc}] audit: {in_audit} -> {out_audit} ({n} lines)")

    # AUTH
    for in_auth in INPUTS[sc].get("auth", []):
        out_auth = output_path_for(in_auth, sc, OUT_DIR)
        n = sanitize_file(in_auth, out_auth, sanitize_file_line := sanitize_auth_line)
        total_auth += n
        print(f"[{sc}] auth:  {in_auth} -> {out_auth} ({n} lines)")

print(f"Done. audit lines: {total_audit}, auth lines: {total_auth}")
print(f"Outputs root: {OUT_DIR.resolve()}")
print(f"Mappings stored in: {MAP_DIR.resolve()}")


[sc3] audit: /Users/zhuoran/Projects/SSCMDataset/data/sc3/audit.log -> /Users/zhuoran/Projects/SSCMDataset/sanidata/sc3/audit.log (206 lines)
[sc3] auth:  /Users/zhuoran/Projects/SSCMDataset/data/sc3/auth.log -> /Users/zhuoran/Projects/SSCMDataset/sanidata/sc3/auth.log (2810 lines)
[sc4] audit: /Users/zhuoran/Projects/SSCMDataset/data/sc4/audit.log -> /Users/zhuoran/Projects/SSCMDataset/sanidata/sc4/audit.log (2491 lines)
[sc4] auth:  /Users/zhuoran/Projects/SSCMDataset/data/sc4/auth.log -> /Users/zhuoran/Projects/SSCMDataset/sanidata/sc4/auth.log (2946 lines)
[sc7] audit: /Users/zhuoran/Projects/SSCMDataset/data/sc7/audit.log -> /Users/zhuoran/Projects/SSCMDataset/sanidata/sc7/audit.log (14763 lines)
Done. audit lines: 17460, auth lines: 5756
Outputs root: /Users/zhuoran/Projects/SSCMDataset/sanidata
Mappings stored in: /Users/zhuoran/Projects/SSCMDataset/mappings


## Quick checks
Sanity-check that `user_map.json` contains **only** the `newt` key.


In [11]:
user_map = load_mapping("user_map")
print("user_map keys:", list(user_map.keys()))
print("user_map:", user_map)


user_map keys: ['newt']
user_map: {'newt': 'USER_615'}


## Preview (first 5 lines)


In [12]:
def head(path: Path, n: int = 5):
    with path.open("r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            print(line.rstrip("\n"))

print("=== audit.sanitized.log ===")
head(OUTPUT_AUDIT, 5)
print("\n=== auth.sanitized.log ===")
head(OUTPUT_AUTH, 5)


=== audit.sanitized.log ===


FileNotFoundError: [Errno 2] No such file or directory: 'audit.sanitized.log'